# Using Neural Networks for sentiment analysis



In [1]:
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/imdb_data.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [4]:
df["sentiment_num"] = df["sentiment"].map(lambda elem: 1 if elem == "positive" else 0)

text_temp, text_test,label_temp, label_test = train_test_split(df["review"], df["sentiment_num"],test_size=0.2,random_state=42,stratify=df["sentiment_num"])

text_train, text_val, label_train, label_val = train_test_split( #validation data split
    text_temp,
    label_temp,
    test_size=0.1,
    random_state=42
)

In [5]:
#normalise the text in the reviews
import re
import string

In [6]:
def normalize(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text
    

In [7]:
normalize(text_train.iloc[0])


'the twins effect chinese action comedy charlene choi gillian chung br br this vampire action comedy is one of my favorites for the very fact that i was thoroughly entertained throughout the entire movie first of all the characters are memorable contributing a myriad of classic scenes charlene and gillian are naturally cute charismatic and humorous this movie was my first exposure to them and all i wanted to do was reach through my television screen and give them a really big hug the remaining cast did well in their supporting roles including jackie chan karen mok the duke josie ho edison chen anthony wong and the vampire bad guys one of which looks eerily familiar to will ferrell even the abominably horrible ekin cheng was good in this one good characters are important of course because they avoid the feeling of boredom by keeping things interesting between action sequences br br and speaking of action this film has plenty of it more importantly there is an emphasis of quality in the 

In [8]:
def tokenize(text):
    text = normalize(text)
    return text.split(" ")


In [9]:
from collections import Counter

In [10]:
def build_vocab(texts, min_freq=5):
    word_counter = Counter()

    for text in texts:
        tokens = tokenize(text)
        word_counter.update(tokens)

    vocab = {}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    index = 2
    
    for word, count in word_counter.items():
        if count >= min_freq:
            vocab[word] = index
            index += 1

    return vocab

In [11]:
def encode(text, vocab):
    tokens = tokenize(text)
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

In [12]:
def pad_sequence(sequence, max_length, pad_index):
    if len(sequence) > max_length:
        return sequence[:max_length]
    return sequence + [pad_index] * (max_length - len(sequence))

def create_mask(sequence, pad_index):
    return [1 if token != pad_index else 0 for token in sequence]

def preprocess_sentence(text, vocab, max_length):
    encoded = encode(text, vocab)
    padded = pad_sequence(encoded, max_length, vocab["<PAD>"])
    mask = create_mask(padded, vocab["<PAD>"])
    return padded, mask

In [13]:
def preprocess_dataset(texts, labels, vocab, max_length):
    all_tokens = []
    all_masks = []
    all_labels = []

    for text, label in zip(texts, labels):
        tokens, mask = preprocess_sentence(text, vocab, max_length)
        all_tokens.append(tokens)
        all_masks.append(mask)
        all_labels.append(label)

    return all_tokens, all_masks, all_labels


In [14]:
vocab = build_vocab(text_train, min_freq=5)


In [15]:
def to_tensors(tokens, masks, labels):
    tokens = torch.tensor(tokens, dtype=torch.long)
    masks = torch.tensor(masks, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return tokens, masks, labels

In [16]:
import torch
from torch.utils.data import Dataset

class SentimentDataset(Dataset):

    def __init__(self, tokens, masks, labels):

        self.tokens = tokens
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "tokens": self.tokens[index],  
            "mask": self.masks[index],      
            "label": self.labels[index]     
        }

In [17]:
import torch
import torch.nn as nn

class SentimentModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()


        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )


        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, tokens, mask):

        embedded = self.embedding(tokens)


        lstm_output, _ = self.lstm(embedded)

        sequence_lengths = mask.sum(dim=1) - 1
        sequence_lengths = torch.clamp(sequence_lengths, min=0)

        batch_indices = torch.arange(
            lstm_output.size(0),
            device=lstm_output.device
        )

        last_outputs = lstm_output[batch_indices, sequence_lengths]

        logits = self.fc(last_outputs)
        return logits


In [18]:
MAX_LENGTH = 200

train_tokens, train_masks, train_labels = preprocess_dataset(
    text_train,
    label_train,
    vocab,
    MAX_LENGTH
)

val_tokens, val_masks, val_labels = preprocess_dataset(
    text_val,
    label_val,
    vocab,
    MAX_LENGTH
)

train_tokens_tensor, train_masks_tensor, train_labels_tensor = to_tensors(
    train_tokens,
    train_masks,
    train_labels
)

val_tokens_tensor, val_masks_tensor, val_labels_tensor = to_tensors(
    val_tokens,
    val_masks,
    val_labels
)

train_dataset = SentimentDataset(
    train_tokens_tensor,
    train_masks_tensor,
    train_labels_tensor
)

val_dataset = SentimentDataset(
    val_tokens_tensor,
    val_masks_tensor,
    val_labels_tensor
)


BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)



In [19]:
model = SentimentModel(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128
)

model = model.to(device)

In [20]:
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [21]:
def train_step(model, batch, criterion, optimizer):
    model.train()

    tokens = batch["tokens"].to(device)
    mask = batch["mask"].to(device)
    labels = batch["label"].float().to(device)

    optimizer.zero_grad()

    outputs = model(tokens, mask).squeeze(1)

    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    return loss.item()


In [22]:
def eval_step(model, batch, criterion):
    model.eval()

    with torch.no_grad():
        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["label"].float().to(device)

        outputs = model(tokens, mask).squeeze(1)
        loss = criterion(outputs, labels)

    return loss.item(), outputs

In [24]:
EPOCHS = 100

for epoch in range(EPOCHS):
    train_loss = 0.0

    for batch in train_loader:
        loss = train_step(model, batch, criterion, optimizer)
        train_loss += loss

    train_loss /= len(train_loader)

    val_loss = 0.0
    correct = 0
    total = 0

    for batch in val_loader:
        loss, outputs = eval_step(model, batch, criterion)
        val_loss += loss

        predictions = (torch.sigmoid(outputs) > 0.5).long()
        labels = batch["label"].to(device)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    val_loss /= len(val_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {accuracy:.4f}"
    )


Epoch 1/100 | Train Loss: 0.5890 | Val Loss: 0.4496 | Val Accuracy: 0.8010
Epoch 2/100 | Train Loss: 0.3513 | Val Loss: 0.3481 | Val Accuracy: 0.8397
Epoch 3/100 | Train Loss: 0.2544 | Val Loss: 0.3256 | Val Accuracy: 0.8595
Epoch 4/100 | Train Loss: 0.1929 | Val Loss: 0.3591 | Val Accuracy: 0.8655
Epoch 5/100 | Train Loss: 0.1422 | Val Loss: 0.3731 | Val Accuracy: 0.8662
Epoch 6/100 | Train Loss: 0.1011 | Val Loss: 0.4550 | Val Accuracy: 0.8605
Epoch 7/100 | Train Loss: 0.0707 | Val Loss: 0.4896 | Val Accuracy: 0.8632
Epoch 8/100 | Train Loss: 0.0488 | Val Loss: 0.5575 | Val Accuracy: 0.8572
Epoch 9/100 | Train Loss: 0.0341 | Val Loss: 0.6136 | Val Accuracy: 0.8542
Epoch 10/100 | Train Loss: 0.0223 | Val Loss: 0.7432 | Val Accuracy: 0.8530
Epoch 11/100 | Train Loss: 0.0205 | Val Loss: 0.7122 | Val Accuracy: 0.8555
Epoch 12/100 | Train Loss: 0.0203 | Val Loss: 0.7589 | Val Accuracy: 0.8550
Epoch 13/100 | Train Loss: 0.0096 | Val Loss: 0.8955 | Val Accuracy: 0.8540
Epoch 14/100 | Train 

In [ ]:

print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())


In [25]:
wrong_predictions = []

model.eval()

with torch.no_grad():

    for batch in val_loader:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(tokens, mask).squeeze(1)
        probabilities = torch.sigmoid(logits)

        predictions = (probabilities > 0.5).long()

        for i in range(len(labels)):
            if predictions[i] != labels[i]:
                wrong_predictions.append({
                    "tokens": tokens[i].cpu(),
                    "true_label": labels[i].item(),
                    "predicted_label": predictions[i].item(),
                    "confidence": probabilities[i].item()
                })


In [26]:
inverse_vocab = {index: word for word, index in vocab.items()}
def decode_tokens(token_ids, inverse_vocab):
    words = []
    for token_id in token_ids:
        if token_id == vocab["<PAD>"]:
            continue
        words.append(inverse_vocab.get(token_id.item(), "<UNK>"))
    return " ".join(words)
for i in range(5):
    example = wrong_predictions[i]

    sentence = decode_tokens(example["tokens"], inverse_vocab)

    print("Sentence:")
    print(sentence)
    print("True label:", "positive" if example["true_label"] == 1 else "negative")
    print("Predicted:", "positive" if example["predicted_label"] == 1 else "negative")
    print("Confidence:", example["confidence"])
    print("-" * 80)


Sentence:
<UNK> <UNK> who co directed the flop <UNK> to hai made his debut in this film br br the film was ahead of it s times in a way though it has a story not to different and it came closest to <UNK> which released 1 week before and luckily this was better and did a better business br br the movie starts off well <UNK> s guilt is well shown at the start though the scene with emraan <UNK> is too crude vulgar br br the scenes between emraan and <UNK> are well handled and the twist in the tale where <UNK> confronts emraan is brilliant br br the pace moves fast and the viewer is kept on the edge but the second girlfriend track of emraan isn t fully convincing br br also the cop track seems half baked br br the finale is too <UNK> too br br direction by <UNK> <UNK> is good music is a winner all songs were fab camera work was stunning br br emraan played his naughty streak very well this was the role that gave him stardom and though he kept playing such roles and got annoying in this fil

## Save model and vocabulary for the API

Run this cell after training to export the model weights and vocabulary so the FastAPI server can load them.

In [ ]:
import json
import os

# Save to the sentiment-analysis-api/models directory
save_dir = "models"
os.makedirs(save_dir, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), os.path.join(save_dir, "sentiment_model.pt"))
print(f"Model saved to {save_dir}/sentiment_model.pt")

# Save vocabulary as JSON
with open(os.path.join(save_dir, "vocab.json"), "w") as f:
    json.dump(vocab, f)
print(f"Vocabulary saved to {save_dir}/vocab.json ({len(vocab)} words)")